# FinMoE — experiment grid on Kaggle

Settings: **Accelerator → GPU T4 x2**, **Internet → on**. Then *Save Version → Save & Run All (Commit)*;
the run continues in the background and the results archive appears in the *Output* tab.

Two job queues run in parallel, one per GPU. `run_grid.py` skips finished runs, so an
interrupted session can simply be committed again.

In [ ]:
import os, subprocess
REPO = "https://github.com/2Gpb/Moe.git"
# for a private repository add a Kaggle secret named GITHUB_TOKEN
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO = REPO.replace("https://", f"https://{token}@")
except Exception:
    pass
if not os.path.exists("/kaggle/working/Moe"):
    subprocess.run(["git", "clone", "-q", REPO, "/kaggle/working/Moe"], check=True)
os.chdir("/kaggle/working/Moe")
!git log --oneline | head -3

In [ ]:
# reuse Kaggle's CUDA build of torch, install the rest of the project with uv
!pip install -q uv
!uv pip install --system -q -e .
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.device_count(), torch.cuda.get_device_name(0))"

In [ ]:
!python -m pytest -q tests

## Experiment queues

Edit `SEEDS` / the queues if only part of the grid is needed. Each queue is balanced to take
roughly the same time.

In [ ]:
SEEDS = ["1", "2"]
QUEUES = {
    "0": ["lora_attn", "lora_experts", "router", "esft_random", "dense_lora_attn", "router_aux"],
    "1": ["esft_token", "esft_gate", "esft_specific", "lora_experts_r1", "lora_attn_router", "esft_least"],
}
procs = []
for gpu, configs in QUEUES.items():
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, PYTHONUNBUFFERED="1")
    log = open(f"/kaggle/working/gpu{gpu}.log", "w")
    cmd = ["python", "scripts/run_grid.py", "--configs", *configs, "--seeds", *SEEDS]
    procs.append(subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT))
codes = [p.wait() for p in procs]
print("exit codes:", codes)

In [ ]:
!grep -h -e FAILED -e '"run"' /kaggle/working/gpu*.log
!tail -n 5 /kaggle/working/gpu*.log

In [ ]:
# light artefacts only; adapter weights of ESFT runs are several hundred MB each
!cd /kaggle/working/Moe && zip -qr /kaggle/working/finmoe_results.zip results/runs
!ls -lh /kaggle/working/finmoe_results.zip

Download `finmoe_results.zip` from the Output tab, unzip it in the repository root and run
`uv run python scripts/make_tables.py` to rebuild the tables and figures with all seeds.